# Real Phase 2 and Phase 3 pilot

Select **Runtime → Change runtime type → T4 GPU**, then run cells in order. This notebook uses public, already collected Phase 1 evidence and frozen source commit `ef90362c287774a9d22ffa05a87f9faa960cad4e`. It needs no API key or paid API calls. The dedicated checkout leaves the original Phase 1 checkout untouched.

Phase 2 runs early answering and probes separately for the original 90 records and the 100-record recovery composite, using each frozen plan's eligibility rules. Phase 3 uses all 20 verified prompt sets, 14 training questions and 6 held-out questions, with **198 fixed generations**. Read `docs/phase2-pilot-plan.json` and `docs/phase3-pilot-plan.json` in the checkout. Allow several hours for Phase 3; Colab can disconnect. Outputs may be invalid or truncated, and human explanation labels stay blank.

**Existing outputs:** Phase 2 refuses any existing output directory and has no automatic resume. Preserve partial evidence before debugging; do not delete it to force a rerun. For a Phase 3 interruption, retain its entire artifact directory and explicitly set `RESUME_PHASE3 = True` below; the runner requires the identical plan and manifest. On reconnect, rerun setup, skip an already completed Phase 2 cell, and resume only Phase 3. If the runtime was replaced, restore your saved evidence to the same artifact paths first. A deliberate new replication needs its own fresh checkout and must be documented separately.

In [ ]:
from pathlib import Path
import os, subprocess, sys

SOURCE_REVISION = "ef90362c287774a9d22ffa05a87f9faa960cad4e"
PROJECT = Path("/content/hint-faithfulness-phase2-phase3-ef90362")
if PROJECT.is_symlink():
    raise RuntimeError("The dedicated checkout must not be a symlink.")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/felmonon/hint-faithfulness.git", str(PROJECT)], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", SOURCE_REVISION], check=True)
actual = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
changes = subprocess.check_output(["git", "-C", str(PROJECT), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
if actual != SOURCE_REVISION or changes:
    raise RuntimeError("Existing checkout differs from the frozen source; preserve it and use a fresh dedicated path.")
os.chdir(PROJECT)
print("Verified source:", actual)

In [ ]:
import json, signal
from datetime import datetime, timezone

# Set GPU visibility before importing torch; child processes inherit this binding.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-pilot.txt"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Choose a T4 GPU runtime before continuing.")
gpu = torch.cuda.get_device_properties(0)
if "T4" not in gpu.name or gpu.total_memory < 14 * 1024**3:
    raise RuntimeError("This notebook fixes the pilot hardware to a T4 with at least 14 GiB GPU memory.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True, timeout=30)

METADATA = PROJECT / "artifacts/pilot-20-phase2-phase3-notebook-v1"
METADATA.mkdir(parents=True, exist_ok=True)
def stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
sys.path.insert(0, str(PROJECT))
from src.provenance import capture
with (METADATA / f"setup-{stamp()}.json").open("x", encoding="utf-8") as stream:
    json.dump({"source_revision": SOURCE_REVISION, "cuda_visible_devices": "0", "runtime": capture()}, stream, indent=2)

def run_logged(command, stage):
    # Keep active logs outside runner output directories: runner checksums then remain stable.
    with (METADATA / f"{stage}-{stamp()}.log").open("x", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            if process.wait() != 0:
                raise RuntimeError(f"{stage} stopped; preserve its log and partial outputs before debugging.")
        except BaseException:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
            raise
print("GPU and runtime recorded. No model experiment has started yet.")

In [ ]:
# Phase 2 has no automatic resume. Skip this cell if Phase 2 already completed.
PHASE2 = PROJECT / "artifacts/pilot-20-phase2-v1"
if PHASE2.exists():
    raise FileExistsError("Phase 2 outputs already exist. Preserve them; do not silently rerun.")
run_logged([sys.executable, "-u", "scripts/run_phase2_pilot.py"], "phase2")

In [ ]:
# Change only this switch to resume an interrupted run with its entire saved output directory.
RESUME_PHASE3 = False
PHASE3 = PROJECT / "artifacts/pilot-20-phase3-v1"
command = [sys.executable, "-u", "scripts/run_phase3_pilot.py"]
if RESUME_PHASE3:
    command.append("--resume")
run_logged(command, "phase3-resume" if RESUME_PHASE3 else "phase3")

In [ ]:
import hashlib, zipfile
from google.colab import files

PHASE2 = PROJECT / "artifacts/pilot-20-phase2-v1"
PHASE3 = PROJECT / "artifacts/pilot-20-phase3-v1"
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

# These completion checks do not require every generated answer to be valid.
for root in (PHASE2, PHASE3):
    checksum_path = root / "checksums.sha256"
    lines = checksum_path.read_text().splitlines()
    if not lines:
        raise RuntimeError(f"No completed artifact inventory in {root.name}.")
    for line in lines:
        expected, relative = line.split("  ", 1)
        path = root / relative
        if path.is_symlink() or not path.resolve().is_relative_to(root.resolve()) or sha256(path) != expected:
            raise RuntimeError(f"Artifact verification failed: {relative}")
completion = json.loads((PHASE3 / "completion.json").read_text())
rows = [json.loads(line) for line in (PHASE3 / "steering.jsonl").read_text().splitlines() if line.strip()]
if completion["status"] != "complete_fixed_sweep" or completion["records"] != 198 or len(rows) != 198 or len({r["record_id"] for r in rows}) != 198:
    raise RuntimeError("Phase 3 is not a complete 198-record sweep; preserve partial evidence separately.")

# Export only these known, non-secret artifact roots. Never archive the checkout, home, or environment files.
roots = [PROJECT / "artifacts/pilot-20-qwen3-1.7b", PHASE2, PHASE3, METADATA]
allowed_suffixes = {".json", ".jsonl", ".csv", ".md", ".txt", ".log", ".npz", ".npy", ".png", ".sha256"}
export_files = {}
for root in roots:
    if root.is_symlink():
        raise RuntimeError("Artifact roots must not be symlinks.")
    for path in sorted(root.rglob("*")):
        if path.is_symlink():
            raise RuntimeError("Refusing a symlink in the evidence export.")
        if path.is_file():
            if path.name.startswith(".") or path.suffix not in allowed_suffixes:
                raise RuntimeError(f"Unexpected artifact filename: {path.name}")
            export_files[str(path.relative_to(PROJECT))] = path
manifest = {"source_revision": SOURCE_REVISION, "created_at_utc": stamp(),
            "files": {name: {"sha256": sha256(path), "bytes": path.stat().st_size} for name, path in export_files.items()},
            "note": "This manifest excludes itself; the ZIP SHA256 is reported separately."}
archive_path = Path("/content") / f"pilot-20-phase2-phase3-{stamp()}.zip"
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for name, path in export_files.items():
        archive.write(path, name)
    archive.writestr("export-manifest.json", json.dumps(manifest, indent=2) + "\n")
archive_sha256 = sha256(archive_path)
print("Evidence archive:", archive_path.name, "bytes:", archive_path.stat().st_size)
print("ZIP SHA256:", archive_sha256)
files.download(str(archive_path))

## Optional backup when the browser download fails

The ZIP still exists after a failed download. The last cell emits a **bounded batch** of its binary content as structured base64 with archive and chunk SHA256 values. Set `EMIT_BACKUP = True`, run one batch, then use **File → Save a copy in Drive** with outputs included. Keep a separately named copy for every batch; rerunning a cell replaces its previous output. Advance `BACKUP_START_CHUNK` by the emitted chunk count until all chunks are saved. Avoid editing or opening the live result files during collection.

One batch is not a complete backup. Recovery must collect every unique chunk index, verify each chunk hash, join decoded chunks in index order, then verify the full ZIP size and SHA256 before using it. This cell exports only the verified archive above, never credentials or arbitrary files. Notebook output limits can vary: confirm each saved copy contains the complete JSON batch. This fallback requires no Drive mount or API key.

In [ ]:
import base64, math
from IPython.display import JSON, display

EMIT_BACKUP = False
BACKUP_START_CHUNK = 0
CHUNK_BYTES = 512 * 1024
CHUNKS_PER_BATCH = 4  # At most 2 MiB binary per output batch; reduce if outputs truncate.
if EMIT_BACKUP:
    if not (1 <= CHUNKS_PER_BATCH <= 4):
        raise ValueError("Use 1 to 4 chunks per batch.")
    if sha256(archive_path) != archive_sha256:
        raise RuntimeError("The verified evidence archive changed.")
    total_bytes = archive_path.stat().st_size
    total_chunks = math.ceil(total_bytes / CHUNK_BYTES)
    if not (0 <= BACKUP_START_CHUNK < total_chunks):
        raise ValueError("BACKUP_START_CHUNK is outside the archive.")
    chunks = []
    with archive_path.open("rb") as stream:
        for index in range(BACKUP_START_CHUNK, min(total_chunks, BACKUP_START_CHUNK + CHUNKS_PER_BATCH)):
            stream.seek(index * CHUNK_BYTES)
            block = stream.read(CHUNK_BYTES)
            chunks.append({"index": index, "bytes": len(block), "sha256": hashlib.sha256(block).hexdigest(),
                           "base64": base64.b64encode(block).decode("ascii")})
    payload = {"schema": "pilot-evidence-zip-chunks-v1", "archive_name": archive_path.name,
               "archive_sha256": archive_sha256, "archive_bytes": total_bytes,
               "chunk_bytes": CHUNK_BYTES, "total_chunks": total_chunks,
               "next_start_chunk": BACKUP_START_CHUNK + len(chunks), "chunks": chunks}
    display(JSON(payload, expanded=False))
    print(f"Chunks {BACKUP_START_CHUNK}–{BACKUP_START_CHUNK + len(chunks) - 1} of {total_chunks}; "
          f"{sum(chunk['bytes'] for chunk in chunks)} binary bytes. Save this batch's notebook copy in Drive.")
else:
    print("Optional backup is off. Enable only after creating the verified ZIP.")